#Multi Percentage Change Map Layers

In [2]:
# 🌐 Install required packages
!pip install folium geopandas pandas

# 📚 Import libraries
import folium
import geopandas as gpd
import pandas as pd
import branca.colormap as cm
from google.colab import drive
from google.colab import files

# 🔐 Mount Google Drive
drive.mount('/content/drive')

# 📁 Define file paths
colorado_geojson_path = '/content/drive/My Drive/colorado_total_population_percent_change_1985_2023_map.geojson'
pctchange_geojson_path = '/content/drive/My Drive/pctchange_gdf.geojson'
dis_avg_geojson_path = '/content/drive/My Drive/dis_avg_percent_change.geojson'
swe_geojson_path = '/content/drive/My Drive/swe_per_change.geojson'

# 📥 Load GeoJSON files
gdf_pop = gpd.read_file(colorado_geojson_path)
gdf_ag = gpd.read_file(pctchange_geojson_path)
gdf_dis = gpd.read_file(dis_avg_geojson_path)
gdf_swe = gpd.read_file(swe_geojson_path)

# 🧹 Ensure numeric columns
gdf_pop['PERCENT_CHANGE_1985_2023'] = pd.to_numeric(gdf_pop['PERCENT_CHANGE_1985_2023'], errors='coerce')
gdf_ag['ia_pctchange'] = pd.to_numeric(gdf_ag['ia_pctchange'], errors='coerce')
gdf_dis['PercentChanges'] = pd.to_numeric(gdf_dis['PercentChanges'], errors='coerce')
gdf_swe['PercentChange'] = pd.to_numeric(gdf_swe['PercentChange'], errors='coerce')

# 🚿 Drop NaNs
gdf_pop.dropna(subset=['PERCENT_CHANGE_1985_2023'], inplace=True)
gdf_ag.dropna(subset=['ia_pctchange'], inplace=True)
gdf_dis.dropna(subset=['PercentChanges'], inplace=True)
gdf_swe.dropna(subset=['PercentChange'], inplace=True)

# 🧾 Update SWE percent changes for selected stations
station_updates = {
    'Middle Fork Camp': -25.0,
    'Buckskin Joe': -26.5,
    'Jones Pass': -11.7,
    'Echo Lake': -23.9,
    'Copeland Lake': -22.2
}
for station, value in station_updates.items():
    gdf_swe.loc[gdf_swe['station_name'].str.contains(station, case=False), 'PercentChange'] = value

# 🗺️ Create base map
m = folium.Map(location=[39.0, -105.5], zoom_start=6, zoom_control=False)

# 🎨 Define colormaps
pop_vmax = min(gdf_pop['PERCENT_CHANGE_1985_2023'].max(), 150)
pop_colormap = cm.LinearColormap(
    ['#ffffb2','#fed976','#feb24c','#fd8d3c','#f03b20','#bd0026'],
    vmin=gdf_pop['PERCENT_CHANGE_1985_2023'].min(),
    vmax=pop_vmax,
    caption='Total Population Percent Change 1985-2001 vs 2002-2024'
)

ag_colormap = cm.LinearColormap(
    ['#005a32','#238443','#41ab5d','#78c679','#addd8e','#d9f0a3','#f7fcb9'],
    vmin=gdf_ag['ia_pctchange'].min(),
    vmax=gdf_ag['ia_pctchange'].max(),
    caption='Irrigated Agriculture Percent Change Pre 2000 VS Post 2000'
)

dis_colormap = cm.LinearColormap(
    ['#4a1486','#6a51a3','#807dba','#9e9ac8','#bcbddc','#dadaeb','#efedf5'],
    vmin=gdf_dis['PercentChanges'].min(),
    vmax=gdf_dis['PercentChanges'].max(),
    caption='Stream Flow Discharge Avg Percent Change 1980-2001 vs 2002-2024'
)

swe_min = gdf_swe['PercentChange'].min()
swe_max = gdf_swe['PercentChange'].max()
swe_colormap = cm.LinearColormap(
    ['#67000d', '#d7301f', '#fc8d59', '#fef0d9', '#91bfdb', '#4575b4'],
    vmin=swe_min,
    vmax=swe_max,
    caption='Snow Water Equivalency Percent Change 1980-2001 vs 2002-2024'
)

# 🧭 Add data layers
folium.GeoJson(
    gdf_pop,
    name='Total Population Percent Change 1985-2001 vs 2002-2024',
    style_function=lambda feature: {
        'fillColor': pop_colormap(feature['properties']['PERCENT_CHANGE_1985_2023']),
        'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['LABEL', 'PERCENT_CHANGE_1985_2023'],
                                  aliases=['County:', 'Percent Change:'], sticky=True)
).add_to(m)
m.add_child(pop_colormap)

folium.GeoJson(
    gdf_ag,
    name='Irrigated Agriculture Percent Change Pre 2000 VS Post 2000',
    style_function=lambda feature: {
        'fillColor': ag_colormap(feature['properties']['ia_pctchange']),
        'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['COUNTY', 'ia_pctchange'],
                                  aliases=['County:', 'Ag % Change:'], sticky=True)
).add_to(m)
m.add_child(ag_colormap)

folium.GeoJson(
    gdf_dis,
    name='Stream Flow Discharge Avg Percent Change 1980-2001 vs 2002-2024',
    style_function=lambda feature: {
        'fillColor': dis_colormap(feature['properties']['PercentChanges']),
        'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['location', 'PercentChanges'],
                                  aliases=['Location:', 'Discharge % Change:'], sticky=True)
).add_to(m)
m.add_child(dis_colormap)

folium.GeoJson(
    gdf_swe,
    name='Snow Water Equivalency Percent Change 1980-2001 vs 2002-2024',
    style_function=lambda feature: {
        'fillColor': swe_colormap(feature['properties']['PercentChange']),
        'color': 'black', 'weight': 0.5, 'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(fields=['station_name', 'PercentChange'],
                                  aliases=['Station:', 'SWE % Change:'], sticky=True)
).add_to(m)
m.add_child(swe_colormap)

# 🧩 Layer control
folium.LayerControl(collapsed=True).add_to(m)

# 📌 Legend CSS
legend_css = """
<style>
    div#legend_0, div#legend_1, div#legend_2, div#legend_3 {
        position: absolute !important;
        bottom: 10px !important;
        left: 10px !important;
        z-index: 9999 !important;
        background-color: #f9f9f9;
        padding: 6px;
        border: 1px solid #ccc;
        margin-bottom: 8px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
        font-size: 16px !important;
    }
</style>
"""
m.get_root().html.add_child(folium.Element(legend_css))

# ✅ Select/Deselect JS
select_js = """
<script>
    function toggleAllLayers(selectAll=true) {
        const checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type="checkbox"]');
        checkboxes.forEach(cb => {
            if (cb.checked !== selectAll) cb.click();
        });
    }
    const controlDiv = document.querySelector('.leaflet-control-layers');
    const btnContainer = document.createElement('div');
    btnContainer.innerHTML = `
        <button onclick="toggleAllLayers(true)" style="margin:2px;">Select All</button>
        <button onclick="toggleAllLayers(false)" style="margin:2px;">Deselect All</button>
    `;
    controlDiv.appendChild(btnContainer);
</script>
"""
m.get_root().html.add_child(folium.Element(select_js))

# 💾 Save and download
output_html = 'colorado_changes_map.html'
m.save(output_html)
files.download(output_html)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>